<a href="https://colab.research.google.com/github/dtc-oss/python_hw/blob/hw_8/hw8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Работа с публичным API и PostgreSQL

В ноутбуке выполняются запросы к [An API of Ice and Fire](https://github.com/joakimskoog/AnApiOfIceAndFire/wiki) и к публичной PostgreSQL-базе [RNAcentral](https://www.rnacentral.org/help/public-database).

Сначала загружаются все книги и дома Вестероса с учётом пагинации, затем формируются датафреймы. Вторая часть подключается к PostgreSQL и получает 10 строк из `rnc_database`.


## 1. Импорт библиотек

In [1]:
%pip install requests pandas psycopg2-binary

In [2]:
import requests
import pandas as pd
import psycopg2

API_BASE = "https://www.anapioficeandfire.com/api"
PAGE_SIZE = 50  # максимальный размер страницы согласно документации API
TIMEOUT = 30


## 2. Вспомогательная функция для пагинации

API возвращает данные постранично. Параметр `pageSize` может быть не больше 50, поэтому последовательно запрашиваем страницы, пока не получим неполную страницу или пустой ответ. `raise_for_status()` сообщит об HTTP-ошибке.

In [3]:
def get_all_pages(endpoint, params=None):
    data_all = []
    page = 1
    base_params = dict(params or {})

    with requests.Session() as session:
        while True:
            query_params = {**base_params, "page": page, "pageSize": PAGE_SIZE}
            response = session.get(endpoint, params=query_params, timeout=TIMEOUT)
            response.raise_for_status()
            page_data = response.json()

            if not isinstance(page_data, list):
                raise ValueError(f"Ожидался список JSON в ответе {response.url}")
            if not page_data:
                break

            data_all.extend(page_data)
            if len(page_data) < PAGE_SIZE:
                break
            page += 1

    return data_all


## 3. Получить информацию обо всех книгах

In [4]:
books_data = get_all_pages(f"{API_BASE}/books")
df_books = pd.json_normalize(books_data)

print("Количество книг:", len(df_books))
display(df_books)


Количество книг: 12


,url,name,isbn,authors,numberOfPages,publisher,country,mediaType,released,characters,povCharacters
0,https://www.anapioficeandfire.com/api/books/1,A Game of Thrones,978-0553103540,[George R. R. Martin],694,Bantam Books,United States,Hardcover,1996-08-01T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
1,https://www.anapioficeandfire.com/api/books/2,A Clash of Kings,978-0553108033,[George R. R. Martin],768,Bantam Books,United States,Hardback,1999-02-02T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
2,https://www.anapioficeandfire.com/api/books/3,A Storm of Swords,978-0553106633,[George R. R. Martin],992,Bantam Books,United States,Hardcover,2000-10-31T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
3,https://www.anapioficeandfire.com/api/books/4,The Hedge Knight,978-0976401100,[George R. R. Martin],164,Dabel Brothers Publishing,United States,GraphicNovel,2005-03-09T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
4,https://www.anapioficeandfire.com/api/books/5,A Feast for Crows,978-0553801507,[George R. R. Martin],784,Bantam Books,United Status,Hardcover,2005-11-08T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
5,https://www.anapioficeandfire.com/api/books/6,The Sworn Sword,978-0785126508,[George R. R. Martin],152,Marvel,United States,Hardcover,2008-06-18T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
6,https://www.anapioficeandfire.com/api/books/7,The Mystery Knight,978-0765360267,[George R. R. Martin],416,Tor Fantasy,United States,Paperback,2011-03-29T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
7,https://www.anapioficeandfire.com/api/books/8,A Dance with Dragons,978-0553801477,[George R. R. Martin],1040,Bantam Books,United States,Hardcover,2011-07-12T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
8,https://www.anapioficeandfire.com/api/books/9,The Princess and the Queen,978-0765332066,[George R. R. Martin],784,Tor Books,United States,Hardcover,2013-12-03T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[]
9,https://www.anapioficeandfire.com/api/books/10,The Rogue Prince,978-0345537263,[George R. R. Martin],832,Bantam Books,United States,Hardcover,2014-06-17T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[]


## 4. Получить информацию обо всех домах Вестероса

In [5]:
houses_data = get_all_pages(f"{API_BASE}/houses")
df_houses = pd.json_normalize(houses_data)

print("Количество домов:", len(df_houses))
display(df_houses)


Количество домов: 444


,url,name,region,coatOfArms,words,titles,seats,currentLord,heir,overlord,founded,founder,diedOut,ancestralWeapons,cadetBranches,swornMembers
0,https://www.anapioficeandfire.com/api/houses/1,House Algood,The Westerlands,"A golden wreath, on a blue field with a gold b...",,[],[],,,https://www.anapioficeandfire.com/api/houses/229,,,,[],[],[]
1,https://www.anapioficeandfire.com/api/houses/2,House Allyrion of Godsgrace,Dorne,"Gyronny Gules and Sable, a hand couped Or",No Foe May Pass,[],[Godsgrace],https://www.anapioficeandfire.com/api/characte...,https://www.anapioficeandfire.com/api/characte...,https://www.anapioficeandfire.com/api/houses/285,,,,[],[],[https://www.anapioficeandfire.com/api/charact...
2,https://www.anapioficeandfire.com/api/houses/3,House Amber,The North,,,[],[],,,,,,,[],[],[]
3,https://www.anapioficeandfire.com/api/houses/4,House Ambrose,The Reach,"Or, semy of ants gules",Never Resting,[],[],https://www.anapioficeandfire.com/api/characte...,,https://www.anapioficeandfire.com/api/houses/398,,,,[],[],[https://www.anapioficeandfire.com/api/charact...
4,https://www.anapioficeandfire.com/api/houses/5,House Appleton of Appleton,The Reach,"Or, an apple tree eradicated proper fructed gu...",,[],[Appleton],,,https://www.anapioficeandfire.com/api/houses/398,,,,[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439,https://www.anapioficeandfire.com/api/houses/440,House Wythers,The Reach,a grey squirrel on white with red border,,[],[],,,https://www.anapioficeandfire.com/api/houses/398,,,,[],[],[https://www.anapioficeandfire.com/api/charact...
440,https://www.anapioficeandfire.com/api/houses/441,House Yarwyck,The Westerlands,Two bronze halberds crossed between four red d...,,[],[],,,https://www.anapioficeandfire.com/api/houses/229,,,,[],[],[https://www.anapioficeandfire.com/api/charact...
441,https://www.anapioficeandfire.com/api/houses/442,House Yelshire,The Reach,"Barry vert and Or, on a chief sable a batterin...",,[],[],,,https://www.anapioficeandfire.com/api/houses/398,,,,[],[],[]
442,https://www.anapioficeandfire.com/api/houses/443,House Yew,The Westerlands,A golden longbow between two crimson flaunches...,,[],[],https://www.anapioficeandfire.com/api/characte...,,https://www.anapioficeandfire.com/api/houses/229,,https://www.anapioficeandfire.com/api/characte...,,[],[],[https://www.anapioficeandfire.com/api/charact...


## 5. Дома с девизом

В документации поле `words` содержит девиз дома, а параметр `hasWords=true` фильтрует дома по наличию девиза. В подсказке задания указан `hasTitles`: он относится к наличию титулов (`titles`), а не девиза. Ниже выполнены оба запроса, чтобы отличие было видно и подсказка тоже была разобрана.

In [6]:
# Дома, у которых заполнен девиз (words)
houses_with_words_data = get_all_pages(
    f"{API_BASE}/houses",
    params={"hasWords": "true"}
)
df_houses_with_words = pd.json_normalize(houses_with_words_data)

print("Домов с девизом:", len(df_houses_with_words))
display(df_houses_with_words[["name", "words", "url"]])


Домов с девизом: 68


,name,words,url
0,House Allyrion of Godsgrace,No Foe May Pass,https://www.anapioficeandfire.com/api/houses/2
1,House Ambrose,Never Resting,https://www.anapioficeandfire.com/api/houses/4
2,House Arryn of the Eyrie,As High as Honor,https://www.anapioficeandfire.com/api/houses/7
3,House Ashford of Ashford,Our Sun Shines Bright,https://www.anapioficeandfire.com/api/houses/8
4,House Baratheon of Storm's End,Ours is the Fury,https://www.anapioficeandfire.com/api/houses/17
...,...,...,...
63,House Wensington,Sound The Charge,https://www.anapioficeandfire.com/api/houses/423
64,House Westerling of the Crag,"Honor, Not Honors",https://www.anapioficeandfire.com/api/houses/425
65,House Wode,Touch Me Not,https://www.anapioficeandfire.com/api/houses/430
66,House Wydman,Right Conquers Might,https://www.anapioficeandfire.com/api/houses/436


In [7]:
# Дома, у которых есть титулы (hasTitles)
houses_with_titles_data = get_all_pages(
    f"{API_BASE}/houses",
    params={"hasTitles": "true"}
)
df_houses_with_titles = pd.json_normalize(houses_with_titles_data)

print("Домов с титулами:", len(df_houses_with_titles))
display(df_houses_with_titles[["name", "titles", "url"]])


Домов с титулами: 165


,name,titles,url
0,House Arryn of the Eyrie,"[King of Mountain and Vale (formerly), Lord of...",https://www.anapioficeandfire.com/api/houses/7
1,House Ashford of Ashford,[Lord of Ashford],https://www.anapioficeandfire.com/api/houses/8
2,House Baelish of Harrenhal,"[Lord Paramount of the Trident, Lord of Harren...",https://www.anapioficeandfire.com/api/houses/10
3,House Banefort of Banefort,[Lord of Banefort],https://www.anapioficeandfire.com/api/houses/13
4,House Bar Emmon of Sharp Point,[Lord of Sharp Point],https://www.anapioficeandfire.com/api/houses/14
...,...,...,...
160,House Whent of Harrenhal,[Lord of Harrenhal],https://www.anapioficeandfire.com/api/houses/427
161,House Wull,"[The Wull, Lord Wull]",https://www.anapioficeandfire.com/api/houses/435
162,House Wylde of Rain House,[Lord of the Rain House],https://www.anapioficeandfire.com/api/houses/438
163,House Wynch of Iron Holt,[Lord of Iron Holt],https://www.anapioficeandfire.com/api/houses/439


## 6. Подключение к PostgreSQL RNAcentral

Библиотека `psycopg2-binary` устанавливается в первой ячейке. Параметры подключения указаны на [официальной странице публичной БД RNAcentral](https://www.rnacentral.org/help/public-database).


In [8]:
DB_CONFIG = {
    "host": "hh-pgsql-public.ebi.ac.uk",
    "port": 5432,
    "dbname": "pfmegrnargs",
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "connect_timeout": 10,
}

conn = psycopg2.connect(**DB_CONFIG)
print("Подключение к PostgreSQL установлено")


Подключение к PostgreSQL установлено


## 7. Получить 10 строк из `rnc_database` и сохранить в DataFrame

In [9]:
try:
    with conn.cursor() as cursor:
        cursor.execute("SELECT * FROM rnc_database LIMIT 10;")
        rows = cursor.fetchall()
        columns = [column[0] for column in cursor.description]

    df_rnc_database = pd.DataFrame(rows, columns=columns)
    display(df_rnc_database)
finally:
    conn.close()


,id,timestamp,userstamp,descr,current_release,full_descr,alive,for_release,display_name,project_id,avg_length,min_length,max_length,num_sequences,num_organisms,description,url,example,reference,last_import_date
0,5,2017-05-17 00:00:00.000000,RNACEN,VEGA,98,VEGA,N,,VEGA,PRJEB4568,NaN,NaN,NaN,0,0,is a repository for high-quality gene models p...,http://vega.sanger.ac.uk/,"[{'upi': 'URS00000B15DA', 'taxid': 9606}, {'up...",[{'title': 'The GENCODE v7 catalog of human lo...,NaT
1,26,2017-05-01 00:00:00.000000,RNACEN,GENCODE,450,GENCODE,N,,GENCODE,,889.0,32.0,205012.0,47677,2,produces high quality reference gene annotatio...,http://gencodegenes.org/,"[{'upi': 'URS00000B15DA', 'taxid': 9606}, {'up...",[{'title': 'GENCODE: the reference human genom...,NaT
2,25,2017-05-11 00:00:00.000000,RNACEN,ENSEMBL,1118,Ensembl,Y,,Ensembl,,1716.0,11.0,347561.0,3373550,267,is a genome browser for vertebrate genomes and...,http://ensembl.org/,"[{'upi': 'URS000025784F', 'taxid': 9606}, {'up...","[{'title': 'Ensembl 2017', 'authors': 'Aken BL...",2026-08-13 23:20:49.325656
3,1,2017-05-01 00:00:00.000000,RNACEN,ENA,1161,ENA,Y,,ENA,,412.0,10.0,900074.0,12086180,814855,provides a comprehensive record of the world's...,https://www.ebi.ac.uk/ena/browser/,"[{'upi': 'URS00002D0E0C', 'taxid': 10090}, {'u...",[{'title': 'The European Nucleotide Archive in...,NaT
4,59,2026-04-21 09:56:18.048915,RNACEN,CIRCATLAS,1090,circAtlas,Y,None,circAtlas,None,379.0,20.0,2808.0,2800521,10,: a gateway to 3 million standardized named ve...,https://ngdc.cncb.ac.cn/circatlas/tutorial1.php,None,[{'title': 'circAtlas 3.0: a gateway to 3 mill...,2026-05-03 04:33:18.130379
5,31,2017-05-01 00:00:00.000000,RNACEN,ENSEMBL_PLANTS,1119,Ensembl Plants,Y,,Ensembl Plants,,563.0,12.0,140059.0,167931,69,is a genome browser for plant genomes that com...,https://plants.ensembl.org/,"[{'upi': 'URS0000A77357', 'taxid': 3702}, {'up...",[{'title': 'Ensembl Genomes 2018: an integrate...,2026-08-13 23:23:46.972921
6,32,2017-05-01 00:00:00.000000,RNACEN,LNCBASE,248,LncBase,Y,,LncBase,,22.0,17.0,26.0,1352,2,experimentally verified and computationally pr...,http://www.microrna.gr/LncBase,"[{'upi': 'URS000075EAB0', 'taxid': 9606}, {'up...",[{'title': 'DIANA-LncBase v2: indexing microRN...,2018-10-04 15:46:52.689649
7,7,2017-05-01 00:00:00.000000,RNACEN,LNCRNADB,464,lncRNAdb,N,,lncRNAdb,PRJEB6238,3086.0,61.0,32753.0,62,10,is a database providing comprehensive annotati...,http://lncrnadb.org/,"[{'upi': 'URS00000478B7', 'taxid': 9606}, {'up...",[{'title': 'lncRNAdb: a reference database for...,2020-08-19 21:22:11.589150
8,57,2026-02-11 13:08:17.996559,RNACEN,CIRCPEDIA,1084,CIRCpedia,Y,None,CIRCpedia,None,975.0,11.0,340824.0,2530830,20,: an expanded repository to annotate both circ...,https://bits.fudan.edu.cn/circpediav3,None,[{'title': 'CIRCpedia v3: an interactive datab...,2026-04-13 22:41:17.759747
9,46,2017-05-01 00:00:00.000000,RNACEN,PIRBASE,1154,piRBase,Y,,piRBase,,27.0,15.0,40.0,480870,26,a database of various piRNA associated data to...,http://www.regulatoryrna.org/database/piRNA/,"[{'upi': 'URS0000061ED0', 'taxid': 9606}, {'up...",[{'title': 'piRBase: a comprehensive database ...,2026-09-18 12:20:09.699981


## 8. Получить только заданные столбцы

In [10]:
conn = psycopg2.connect(**DB_CONFIG)
try:
    with conn.cursor() as cursor:
        cursor.execute(
            "SELECT display_name, num_sequences, num_organisms, url "
            "FROM rnc_database LIMIT 10;"
        )
        rows = cursor.fetchall()
        columns = [column[0] for column in cursor.description]

    df_rnc_selected = pd.DataFrame(rows, columns=columns)
    display(df_rnc_selected)
finally:
    conn.close()


,display_name,num_sequences,num_organisms,url
0,VEGA,0,0,http://vega.sanger.ac.uk/
1,GENCODE,47677,2,http://gencodegenes.org/
2,Ensembl,3373550,267,http://ensembl.org/
3,ENA,12086180,814855,https://www.ebi.ac.uk/ena/browser/
4,circAtlas,2800521,10,https://ngdc.cncb.ac.cn/circatlas/tutorial1.php
5,Ensembl Plants,167931,69,https://plants.ensembl.org/
6,LncBase,1352,2,http://www.microrna.gr/LncBase
7,lncRNAdb,62,10,http://lncrnadb.org/
8,CIRCpedia,2530830,20,https://bits.fudan.edu.cn/circpediav3
9,piRBase,480870,26,http://www.regulatoryrna.org/database/piRNA/


## Итог

После выполнения ячеек созданы датафреймы:
- `df_books` — все книги из API;
- `df_houses` — все дома;
- `df_houses_with_words` — дома с девизом;
- `df_houses_with_titles` — дома с титулами (фильтр `hasTitles=true`);
- `df_rnc_database` — 10 строк со всеми столбцами из `rnc_database`;
- `df_rnc_selected` — 10 строк с `display_name`, `num_sequences`, `num_organisms`, `url`.